# **Importer les bibliothèques**



In [1]:
import os
import json
import shutil
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:

from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


# **Préparer les dossiers**

In [3]:

PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/nlp_translation")

RESULTS_DIR = PROJECT_DIR / "results"
EXP4_DIR = RESULTS_DIR / "experience_4"
MODELS_DIR = PROJECT_DIR / "models"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
EXP4_DIR.mkdir(parents=True, exist_ok=True)

print("Dossier résultats :", RESULTS_DIR)
print("Dossier expérience 4 :", EXP4_DIR)


Dossier résultats : /content/drive/MyDrive/Colab Notebooks/nlp_translation/results
Dossier expérience 4 : /content/drive/MyDrive/Colab Notebooks/nlp_translation/results/experience_4


# **Ajouter les trois fichiers de l'expérience 2**


In [4]:

FICHIERS = [
    "lstm_history.json",
    "lstm_metrics.json",
    "lstm_test_translations.csv"
]

for nom in FICHIERS:
    destination = RESULTS_DIR / nom

    if destination.is_file():
        print("OK, déjà présent :", nom)
        continue

    candidats = [
        Path("/content") / nom,
        PROJECT_DIR / nom
    ]

    source = next((p for p in candidats if p.is_file()), None)

    if source is not None:
        shutil.copy2(source, destination)
        print("Copié :", nom)
    else:
        print("MANQUANT :", nom)
        print("À récupérer depuis l'expérience 2.")

OK, déjà présent : lstm_history.json
OK, déjà présent : lstm_metrics.json
OK, déjà présent : lstm_test_translations.csv


--> Cette cellule recherche les fichiers déjà produits. Elle ne fabrique pas de résultats manquants.

# **Vérifier les modèles et les résultats**

In [5]:

for nom in ["lstm_best.pt", "lstm_last.pt"]:
    chemin = MODELS_DIR / nom
    print(nom, ":", "Présent" if chemin.is_file() else "À vérifier sur Drive")

csv_path = RESULTS_DIR / "lstm_test_translations.csv"

if not csv_path.is_file():
    raise FileNotFoundError(
        "Ajoute lstm_test_translations.csv dans results/"
    )

print("CSV trouvé. Prête pour l'analyse.")


lstm_best.pt : À vérifier sur Drive
lstm_last.pt : À vérifier sur Drive
CSV trouvé. Prête pour l'analyse.


# **Expérience 4 — Analyse des longueurs**



```
# Charger les traductions sauvegardées
```



In [6]:

df = pd.read_csv(csv_path)

print("Nombre de traductions :", len(df))
print("Colonnes :", df.columns.tolist())

display(df.head(10))


Nombre de traductions : 2000
Colonnes : ['source', 'reference', 'prediction', 'source_length', 'ended_with_eos']


,source,reference,prediction,source_length,ended_with_eos
0,- vous étiez en train de vous embrasser à l'ar...,- you were at a bus stop kissing him!,- you you you to you you you !,13,True
1,"avec une ironie farceuse, le jeune artiste tch...",with irony and mischief the young czech artist...,in the <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UN...,17,True
2,qui va parler au vendeur de voitures ? big fre...,- who's going to talk to the used car salesman?,"we have to <UNK> <UNK> , <UNK> <UNK> .",11,True
3,on vous dit éclairé : votre musique parle d'hu...,"people think you are a great man, that your mu...","you know , you , you , you , and you , and you .",16,True
4,autres coraux,lace corals,<UNK> <UNK>,2,True
5,"dites, je me demandais...","yeah, there's one man i know who could nail him.","oh , i . . . . .",8,True
6,peter et moi étions dans un accident de voitur...,peter and i were in a car crash.,and i was a to me . . . . . .,12,True
7,ces lois mettent en place une république parle...,these laws establish a parliamentary republic ...,these are are the to of the and and and and the .,15,True
8,"les activités de suivi, de recherche et d'éval...","monitoring, research and evaluation activities...",the and of and and and and and and and and and...,28,True
9,lorsque les actes des institutions adoptés ava...,where acts of the institutions prior to 1 may ...,when the of of the the of the the the the of t...,46,True




```
# Nettoyer et vérifier les données
```



In [7]:

colonnes = ["source", "reference", "prediction"]

for col in colonnes:
    if col not in df.columns:
        raise ValueError(f"Colonne manquante : {col}")

print("Valeurs manquantes :")
print(df[colonnes].isna().sum())

df = df.dropna(subset=colonnes).copy()

for col in colonnes:
    df[col] = df[col].astype(str).str.strip()

df = df[
    (df["source"] != "") &
    (df["reference"] != "") &
    (df["prediction"] != "")
].copy()

print("Traductions exploitables :", len(df))


Valeurs manquantes :
source        0
reference     0
prediction    0
dtype: int64
Traductions exploitables : 2000




```
# Calculer la longueur des phrases
```



In [8]:

df["longueur"] = df["source"].apply(
    lambda phrase: len(phrase.split())
)

display(
    df[["source", "longueur", "prediction"]].head(10)
)

,source,longueur,prediction
0,- vous étiez en train de vous embrasser à l'ar...,12,- you you you to you you you !
1,"avec une ironie farceuse, le jeune artiste tch...",15,in the <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UN...
2,qui va parler au vendeur de voitures ? big fre...,10,"we have to <UNK> <UNK> , <UNK> <UNK> ."
3,on vous dit éclairé : votre musique parle d'hu...,14,"you know , you , you , you , and you , and you ."
4,autres coraux,2,<UNK> <UNK>
5,"dites, je me demandais...",4,"oh , i . . . . ."
6,peter et moi étions dans un accident de voitur...,9,and i was a to me . . . . . .
7,ces lois mettent en place une république parle...,14,these are are the to of the and and and and the .
8,"les activités de suivi, de recherche et d'éval...",26,the and of and and and and and and and and and...
9,lorsque les actes des institutions adoptés ava...,43,when the of of the the of the the the the of t...




```
# Classer les phrases
```



# **Classer les phrases**

In [9]:

def categorie_longueur(n):
    if n <= 10:
        return "Courte"
    elif n <= 20:
        return "Moyenne"
    else:
        return "Longue"

df["categorie"] = df["longueur"].apply(
    categorie_longueur
)

ordre = ["Courte", "Moyenne", "Longue"]

print(df["categorie"].value_counts().reindex(ordre, fill_value=0))

categorie
Courte     866
Moyenne    466
Longue     668
Name: count, dtype: int64


Catégorie |	Nombre de mots

Courte  	| 1 à 10

Moyenne	 | 11 à 20

Longue	  |  Plus de 20



```
# Calculer le score BLEU pour chaque catégorie
```

==> Le score BLEU mesure la similarité entre les traductions prédites et les traductions de référence.



In [10]:

!pip -q install sacrebleu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 10.9 MB/s eta 0:00:00




```
# Tracer le graphique BLEU selon la longueur
```



In [13]:

print("=== REFERENCES ===")
for phrase in df["reference"].head(5):
    print(repr(phrase))

print("\n=== PREDICTIONS ===")
for phrase in df["prediction"].head(5):
    print(repr(phrase))


=== REFERENCES ===
'- you were at a bus stop kissing him!'
'with irony and mischief the young czech artist krištof kintera turns art and life on their heads.'
"- who's going to talk to the used car salesman?"
'people think you are a great man, that your music speaks of humanity, warmth and understanding.'
'lace corals'

=== PREDICTIONS ===
'- you you you to you you you !'
'in the <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> .'
'we have to <UNK> <UNK> , <UNK> <UNK> .'
'you know , you , you , you , and you , and you .'
'<UNK> <UNK>'




```
# Corriger la ponctuation
```



In [14]:

import re

def corriger_ponctuation(texte):
    texte = str(texte).strip()
    texte = re.sub(r"\s+([.,!?;:])", r"\1", texte)
    texte = re.sub(r"\s+", " ", texte)
    return texte

# Les références sont déjà correctement formatées
df["reference_eval"] = df["reference"].copy()

# Corriger seulement les prédictions
df["prediction_eval"] = df["prediction"].apply(
    corriger_ponctuation
)

print("=== AVANT CORRECTION ===")
print(df["prediction"].iloc[0])

print("\n=== APRÈS CORRECTION ===")
print(df["prediction_eval"].iloc[0])


=== AVANT CORRECTION ===
- you you you to you you you !

=== APRÈS CORRECTION ===
- you you you to you you you!




```
# calcul du BLEU
```



In [15]:

import sacrebleu
import pandas as pd

resultats = []

for categorie in ["Courte", "Moyenne", "Longue"]:

    groupe = df[df["categorie"] == categorie]

    if len(groupe) == 0:
        bleu = float("nan")
    else:
        bleu = sacrebleu.corpus_bleu(
            groupe["prediction_eval"].tolist(),
            [groupe["reference_eval"].tolist()]
        ).score

    resultats.append({
        "Categorie": categorie,
        "Nombre": len(groupe),
        "BLEU": round(bleu, 2)
    })

df_bleu = pd.DataFrame(resultats)

display(df_bleu)


,Categorie,Nombre,BLEU
0,Courte,866,2.93
1,Moyenne,466,1.42
2,Longue,668,0.85


In [16]:

def compter_unk(phrase):
    return phrase.lower().count("<unk>")

def contient_repetition(phrase):
    mots = phrase.lower().split()
    return any(
        mots[i] == mots[i+1] == mots[i+2]
        for i in range(len(mots)-2)
    )

df["nb_unk"] = df["prediction"].apply(compter_unk)
df["repetition"] = df["prediction"].apply(contient_repetition)

analyse = df.groupby("categorie").agg(
    nombre_phrases=("source", "size"),
    moyenne_unk=("nb_unk", "mean"),
    pourcentage_repetitions=("repetition", "mean")
).reindex(["Courte", "Moyenne", "Longue"])

analyse["pourcentage_repetitions"] *= 100

display(analyse.round(2))


,nombre_phrases,moyenne_unk,pourcentage_repetitions
categorie,,,
Courte,866,1.32,25.17
Moyenne,466,3.67,47.21
Longue,668,9.67,57.34


In [17]:

df_bleu.to_csv(
    EXP4_DIR / "bleu_par_longueur.csv",
    index=False
)

analyse.to_csv(
    EXP4_DIR / "analyse_erreurs_longueur.csv"
)

print("Résultats sauvegardés avec succès.")


Résultats sauvegardés avec succès.


In [18]:

import re
import numpy as np

def compter_tokens(texte):
    return len(re.findall(
        r"<unk>|\w+(?:['’]\w+)*",
        str(texte).lower()
    ))

df["nb_tokens_pred"] = df["prediction"].apply(
    compter_tokens
)

analyse_unk = df.groupby("categorie").agg(
    total_unk=("nb_unk", "sum"),
    total_tokens=("nb_tokens_pred", "sum")
)

analyse_unk["taux_unk_%"] = (
    100 * analyse_unk["total_unk"]
    / analyse_unk["total_tokens"].replace(0, np.nan)
)

analyse_unk = analyse_unk.reindex(
    ["Courte", "Moyenne", "Longue"]
)

display(analyse_unk.round(2))

analyse_unk.to_csv(
    EXP4_DIR / "taux_unk_par_longueur.csv"
)


,total_unk,total_tokens,taux_unk_%
categorie,,,
Courte,1145,3780,30.29
Moyenne,1709,5455,31.33
Longue,6458,19975,32.33




```
# Afficher des exemples de traduction
```



In [19]:

for categorie in ordre:

    print("\n" + "="*45)
    print("CATÉGORIE :", categorie)
    print("="*45)

    exemples = df[
        df["categorie"] == categorie
    ].head(3)

    for _, ligne in exemples.iterrows():

        print("\nFrançais :", ligne["source"])
        print("Référence :", ligne["reference"])
        print("Prédiction :", ligne["prediction"])
        print("-"*45)



CATÉGORIE : Courte

Français : qui va parler au vendeur de voitures ? big freddy.
Référence : - who's going to talk to the used car salesman?
Prédiction : we have to <UNK> <UNK> , <UNK> <UNK> .
---------------------------------------------

Français : autres coraux
Référence : lace corals
Prédiction : <UNK> <UNK>
---------------------------------------------

Français : dites, je me demandais...
Référence : yeah, there's one man i know who could nail him.
Prédiction : oh , i . . . . .
---------------------------------------------

CATÉGORIE : Moyenne

Français : - vous étiez en train de vous embrasser à l'arrêt de bus!
Référence : - you were at a bus stop kissing him!
Prédiction : - you you you to you you you !
---------------------------------------------

Français : avec une ironie farceuse, le jeune artiste tchèque krištof kintera chamboule l'art et la vie.
Référence : with irony and mischief the young czech artist krištof kintera turns art and life on their heads.
Prédiction : in 



```
# Analyse des trois métriques
```



In [20]:

import sacrebleu
import pandas as pd

resultats = []

for categorie in ["Courte", "Moyenne", "Longue"]:

    groupe = df[df["categorie"] == categorie]

    if len(groupe) == 0:
        continue

    predictions = groupe["prediction_eval"].tolist()
    references = [groupe["reference_eval"].tolist()]

    bleu = sacrebleu.corpus_bleu(
        predictions, references
    ).score

    chrf = sacrebleu.corpus_chrf(
        predictions, references
    ).score

    ter = sacrebleu.corpus_ter(
        predictions, references
    ).score

    resultats.append({
        "Categorie": categorie,
        "BLEU": round(bleu, 2),
        "chrF": round(chrf, 2),
        "TER": round(ter, 2)
    })

comparaison = pd.DataFrame(resultats)

display(comparaison)

comparaison.to_csv(
    EXP4_DIR / "comparaison_metriques.csv",
    index=False
)


,Categorie,BLEU,chrF,TER
0,Courte,2.93,9.10,97.79
1,Moyenne,1.42,8.24,96.39
2,Longue,0.85,8.05,101.53


In [22]:

from google.colab import files
from pathlib import Path
import zipfile

# Dossier contenant les résultats
EXP4_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/nlp_translation/results/experience_4"
)

# Fichiers à télécharger
fichiers = [
    "bleu_par_longueur.csv",
    "comparaison_metriques.csv",
    "analyse_erreurs_longueur.csv",
    "taux_unk_par_longueur.csv",
    "bleu_par_longueur.png"
]

# Créer le ZIP
zip_path = "/content/experience_4_resultats.zip"

with zipfile.ZipFile(zip_path, "w") as zipf:
    for nom in fichiers:
        chemin = EXP4_DIR / nom

        if chemin.exists():
            zipf.write(chemin, arcname=nom)
            print("Ajouté :", nom)
        else:
            print("Fichier introuvable :", nom)

# Télécharger
files.download(zip_path)


Ajouté : bleu_par_longueur.csv
Ajouté : comparaison_metriques.csv
Ajouté : analyse_erreurs_longueur.csv
Ajouté : taux_unk_par_longueur.csv
Ajouté : bleu_par_longueur.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>